In [76]:
import os
import sys
from dotenv import load_dotenv
from sklearn.metrics import adjusted_mutual_info_score
load_dotenv() 

# Set the target folder name you want to reach
target_folder = "NCEAS_Unsupervised_NLP"

# Get the current working directory
current_dir = os.getcwd()

# Loop to move up the directory tree until we reach the target folder
while os.path.basename(current_dir) != target_folder:
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    if parent_dir == current_dir:
        # If we reach the root directory and haven't found the target, exit
        raise FileNotFoundError(f"{target_folder} not found in the directory tree.")
    current_dir = parent_dir

# Change working directory to the project root
os.chdir(current_dir)

# Add the "NCEAS_Unsupervised_NLPt" directory to sys.path
sys.path.insert(0, current_dir)

In [77]:
import pandas as pd

In [78]:
# =========================
# 1. Setup
# =========================
import pandas as pd

# =========================
# 2. Load files
# =========================
comparison = pd.read_csv("src/evaluations/arxiv/results/arxiv_FULL.csv")
hier = pd.read_csv("src/evaluations/arxiv/results/arxiv_hierarchical.csv")

# =========================
# 3. Clean columns
# =========================
comparison.columns = comparison.columns.str.strip()
hier.columns = hier.columns.str.strip()


In [79]:

# 4. Rename columns

comparison = comparison.rename(columns={
    "embedding_model": "embedding",
    "reduction_method": "reduction",
    "cluster_method": "cluster_method"
})


# 5. Merge

df = pd.merge(
    comparison,
    hier,
    on=["embedding", "reduction", "cluster_method"],
    how="left"
)

# 6. Aggregate

final_df = (
    df.groupby(['embedding', 'reduction', 'cluster_method'])[
        ['ARI', 'AMI', 'dendrogram_purity', 'lca_f1']
    ]
    .mean()
    .reset_index()
)

# 7. Clean naming
final_df = final_df.replace({
    "DC": "Diffusion condensation",
    "DiffusionCondensation": "Diffusion condensation",
    "Diffusion Condensation": "Diffusion condensation"
})

# 8. Preview
print("Final DF preview:")
display(final_df.head())

# 9. ARI TABLE (MAIN RESULT)
pivot_ari = final_df.pivot_table(
    index=['reduction', 'cluster_method'],
    columns='embedding',
    values='ARI'
).sort_index()

print("=== ARI TABLE ===")
display(pivot_ari)


# 10. COMBINED HIERARCHICAL TABLE
pivot_hier = final_df.pivot_table(
    index=['reduction', 'cluster_method'],
    columns='embedding',
    values=['lca_f1', 'dendrogram_purity']
).sort_index()

print("=== HIERARCHICAL TABLE (LCA + DENDROGRAM) ===")
display(pivot_hier)

Final DF preview:


,embedding,reduction,cluster_method,ARI,AMI,dendrogram_purity,lca_f1
0,MiniLM,PCA,Agglomerative,-0.000128,-0.000335,0.155833,0.029997
1,MiniLM,PCA,Diffusion condensation,0.000038,-0.000103,0.152467,0.113900
2,MiniLM,PCA,HDBSCAN,0.000511,0.000205,0.157877,0.117815
3,MiniLM,PHATE,Agglomerative,-0.000147,0.000209,0.155200,0.031703
4,MiniLM,PHATE,Diffusion condensation,0.001004,0.000084,0.152000,0.101476


=== ARI TABLE ===


embedding                           MiniLM      Qwen
reduction cluster_method                            
PCA       Agglomerative          -0.000128  0.000008
          Diffusion condensation  0.000038  0.000423
          HDBSCAN                 0.000511  0.000315
PHATE     Agglomerative          -0.000147 -0.000186
          Diffusion condensation  0.001004 -0.000126
          HDBSCAN                 0.000023 -0.001349
UMAP      Agglomerative          -0.000237 -0.000109
          Diffusion condensation -0.000620 -0.000193
          HDBSCAN                 0.000100 -0.001139

=== HIERARCHICAL TABLE (LCA + DENDROGRAM) ===


dendrogram_purity              lca_f1  \
embedding                                   MiniLM      Qwen    MiniLM   
reduction cluster_method                                                 
PCA       Agglomerative                   0.155833  0.155400  0.029997   
          Diffusion condensation          0.152467  0.152000  0.113900   
          HDBSCAN                         0.157877  0.152486  0.117815   
PHATE     Agglomerative                   0.155200  0.154400  0.031703   
          Diffusion condensation          0.152000  0.152000  0.101476   
          HDBSCAN                         0.152014  0.157230  0.115693   
UMAP      Agglomerative                   0.154533  0.156067  0.031283   
          Diffusion condensation          0.152133  0.152000  0.108008   
          HDBSCAN                         0.173638  0.165052  0.015911   

                                            
embedding                             Qwen  
reduction cluster_method                    
PCA       Agglomerative           0.029687  
          Diffusion condensation  0.108938  
          HDBSCAN                 0.116655  
PHATE     Agglomerative           0.032197  
          Diffusion condensation  0.095083  
          HDBSCAN                 0.087338  
UMAP      Agglomerative           0.031770  
          Diffusion condensation  0.107058  
          HDBSCAN                 0.077168

In [80]:
final_df

,embedding,reduction,cluster_method,ARI,AMI,dendrogram_purity,lca_f1
0,MiniLM,PCA,Agglomerative,-0.000128,-0.000335,0.155833,0.029997
1,MiniLM,PCA,Diffusion condensation,0.000038,-0.000103,0.152467,0.113900
2,MiniLM,PCA,HDBSCAN,0.000511,0.000205,0.157877,0.117815
3,MiniLM,PHATE,Agglomerative,-0.000147,0.000209,0.155200,0.031703
4,MiniLM,PHATE,Diffusion condensation,0.001004,0.000084,0.152000,0.101476
5,MiniLM,PHATE,HDBSCAN,0.000023,0.000025,0.152014,0.115693
6,MiniLM,UMAP,Agglomerative,-0.000237,0.000085,0.154533,0.031283
7,MiniLM,UMAP,Diffusion condensation,-0.000620,0.000018,0.152133,0.108008
8,MiniLM,UMAP,HDBSCAN,0.000100,-0.000750,0.173638,0.015911
9,Qwen,PCA,Agglomerative,0.000008,-0.000573,0.155400,0.029687


In [81]:
pivot_ari

embedding                           MiniLM      Qwen
reduction cluster_method                            
PCA       Agglomerative          -0.000128  0.000008
          Diffusion condensation  0.000038  0.000423
          HDBSCAN                 0.000511  0.000315
PHATE     Agglomerative          -0.000147 -0.000186
          Diffusion condensation  0.001004 -0.000126
          HDBSCAN                 0.000023 -0.001349
UMAP      Agglomerative          -0.000237 -0.000109
          Diffusion condensation -0.000620 -0.000193
          HDBSCAN                 0.000100 -0.001139

In [82]:
pivot_hier

dendrogram_purity              lca_f1  \
embedding                                   MiniLM      Qwen    MiniLM   
reduction cluster_method                                                 
PCA       Agglomerative                   0.155833  0.155400  0.029997   
          Diffusion condensation          0.152467  0.152000  0.113900   
          HDBSCAN                         0.157877  0.152486  0.117815   
PHATE     Agglomerative                   0.155200  0.154400  0.031703   
          Diffusion condensation          0.152000  0.152000  0.101476   
          HDBSCAN                         0.152014  0.157230  0.115693   
UMAP      Agglomerative                   0.154533  0.156067  0.031283   
          Diffusion condensation          0.152133  0.152000  0.108008   
          HDBSCAN                         0.173638  0.165052  0.015911   

                                            
embedding                             Qwen  
reduction cluster_method                    
PCA       Agglomerative           0.029687  
          Diffusion condensation  0.108938  
          HDBSCAN                 0.116655  
PHATE     Agglomerative           0.032197  
          Diffusion condensation  0.095083  
          HDBSCAN                 0.087338  
UMAP      Agglomerative           0.031770  
          Diffusion condensation  0.107058  
          HDBSCAN                 0.077168